# 🛒 Notebook 3 — Qdrant with E-commerce Products

> **Goal:** Qdrant's superpower is **rich payload filtering** — perfect for product search.

## 🎯 Learning Goals
- Run Qdrant entirely **in-memory** (zero install)
- Insert points with **payload** (JSON metadata)
- Use **must / should / must_not / range** filters
- Combine vector similarity with strict business rules

## 🍱 Analogy
**Qdrant = Elasticsearch for vectors.**
- Bring-your-own-vectors model.
- Filters are first-class — you can require `price < 50 AND brand IN ("Nike","Adidas") AND in_stock=true` and *then* do the vector search.

In [ ]:
# 📦 Install
# %pip install -q qdrant-client sentence-transformers pandas

In [ ]:
from IPython.display import SVG
SVG("""<svg viewBox="0 0 720 230" xmlns="http://www.w3.org/2000/svg" width="720"><style>text{font-family:Inter,system-ui,sans-serif}</style><rect width="720" height="230" fill="#f8fafc" rx="12"/><rect x="20" y="60" width="140" height="80" rx="10" fill="#dbeafe" stroke="#2563eb"/><text x="90" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#1e3a8a">Products</text><text x="90" y="110" text-anchor="middle" font-size="11" fill="#1e3a8a">20 SKUs</text><text x="90" y="125" text-anchor="middle" font-size="11" fill="#1e3a8a">+ price, brand</text><rect x="200" y="60" width="140" height="80" rx="10" fill="#ede9fe" stroke="#7c3aed"/><text x="270" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#5b21b6">MiniLM</text><text x="270" y="110" text-anchor="middle" font-size="11" fill="#5b21b6">encode title+desc</text><text x="270" y="125" text-anchor="middle" font-size="11" fill="#5b21b6">→ 384-d</text><rect x="380" y="60" width="140" height="80" rx="10" fill="#fef3c7" stroke="#f59e0b"/><text x="450" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#78350f">Qdrant Collection</text><text x="450" y="110" text-anchor="middle" font-size="11" fill="#78350f">in-memory mode</text><text x="450" y="125" text-anchor="middle" font-size="11" fill="#78350f">payload + filters</text><rect x="560" y="60" width="140" height="80" rx="10" fill="#dcfce7" stroke="#15803d"/><text x="630" y="90" text-anchor="middle" font-size="13" font-weight="700" fill="#065f46">Hybrid Search</text><text x="630" y="110" text-anchor="middle" font-size="11" fill="#065f46">semantic + filter</text><text x="630" y="125" text-anchor="middle" font-size="11" fill="#065f46">+ score boost</text><text x="172" y="105" font-size="22" fill="#475569">→</text><text x="352" y="105" font-size="22" fill="#475569">→</text><text x="532" y="105" font-size="22" fill="#475569">→</text><text x="360" y="35" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">Qdrant pipeline</text><text x="360" y="180" text-anchor="middle" font-size="11" fill="#475569">Qdrant payloads = first-class JSON metadata with rich filter operators</text></svg>""")

In [ ]:
# 🛍️ E-commerce catalog
products = [
    {"id": 1,  "name": "Nike Air Zoom Running Shoes",     "brand": "Nike",     "category": "Footwear",   "price": 119.99, "in_stock": True,  "desc": "Lightweight running shoes with responsive foam cushioning for road runners and daily training."},
    {"id": 2,  "name": "Adidas Ultraboost 22",            "brand": "Adidas",   "category": "Footwear",   "price": 179.99, "in_stock": True,  "desc": "Premium running shoes with energy-returning Boost midsole and adaptive Primeknit upper."},
    {"id": 3,  "name": "Puma Suede Classic Sneakers",     "brand": "Puma",     "category": "Footwear",   "price": 79.99,  "in_stock": False, "desc": "Iconic suede casual sneakers with rubber cup outsole, perfect for streetwear looks."},
    {"id": 4,  "name": "Levi's 511 Slim Fit Jeans",       "brand": "Levis",    "category": "Apparel",    "price": 69.50,  "in_stock": True,  "desc": "Slim fit denim jeans with stretch fabric for all-day comfort in dark indigo wash."},
    {"id": 5,  "name": "Uniqlo Heattech Long Sleeve Tee", "brand": "Uniqlo",   "category": "Apparel",    "price": 19.90,  "in_stock": True,  "desc": "Thermal base layer t-shirt that traps body heat, ideal for cold weather layering."},
    {"id": 6,  "name": "Nike Dri-FIT Training Shorts",    "brand": "Nike",     "category": "Apparel",    "price": 34.99,  "in_stock": True,  "desc": "Quick-drying athletic shorts with mesh side panels for breathability during workouts."},
    {"id": 7,  "name": "Sony WH-1000XM5 Headphones",      "brand": "Sony",     "category": "Electronics","price": 399.00, "in_stock": True,  "desc": "Wireless noise-cancelling over-ear headphones with industry-leading ANC and 30-hour battery."},
    {"id": 8,  "name": "Apple AirPods Pro 2",             "brand": "Apple",    "category": "Electronics","price": 249.00, "in_stock": True,  "desc": "True-wireless earbuds with active noise cancellation, transparency mode, and spatial audio."},
    {"id": 9,  "name": "Bose SoundLink Mini Speaker",     "brand": "Bose",     "category": "Electronics","price": 199.00, "in_stock": False, "desc": "Compact portable Bluetooth speaker with rich bass and twelve-hour battery life."},
    {"id": 10, "name": "Kindle Paperwhite e-Reader",      "brand": "Amazon",   "category": "Electronics","price": 139.99, "in_stock": True,  "desc": "Waterproof e-reader with glare-free 6.8-inch display and weeks of battery for novels and books."},
    {"id": 11, "name": "Yeti Rambler Insulated Bottle",   "brand": "Yeti",     "category": "Outdoors",   "price": 45.00,  "in_stock": True,  "desc": "Stainless steel double-wall vacuum insulated water bottle keeps drinks cold for hours."},
    {"id": 12, "name": "Patagonia Nano Puff Jacket",      "brand": "Patagonia","category": "Outdoors",   "price": 229.00, "in_stock": True,  "desc": "Lightweight insulated puffy jacket with recycled polyester shell for hiking and travel."},
    {"id": 13, "name": "Coleman 4-Person Camping Tent",   "brand": "Coleman",  "category": "Outdoors",   "price": 89.99,  "in_stock": True,  "desc": "Easy-setup dome tent for four people with weather-tek system and built-in rainfly."},
    {"id": 14, "name": "Cuisinart Stainless Frying Pan",  "brand": "Cuisinart","category": "Kitchen",    "price": 49.99,  "in_stock": True,  "desc": "Tri-ply stainless steel skillet with even heat distribution and oven-safe handle."},
    {"id": 15, "name": "Nespresso VertuoPlus Machine",    "brand": "Nespresso","category": "Kitchen",    "price": 149.00, "in_stock": True,  "desc": "Single-serve coffee maker brewing espresso and large cup sizes with capsule recognition."},
    {"id": 16, "name": "Instant Pot Duo 6-Quart",         "brand": "InstantPot","category": "Kitchen",   "price": 99.99,  "in_stock": False, "desc": "Multi-functional electric pressure cooker that pressure cooks, slow cooks, sautes and steams."},
    {"id": 17, "name": "Dyson V11 Cordless Vacuum",       "brand": "Dyson",    "category": "Home",       "price": 569.00, "in_stock": True,  "desc": "Cordless stick vacuum with intelligent suction and LCD screen showing remaining run time."},
    {"id": 18, "name": "Philips Hue Smart Bulb 4-Pack",   "brand": "Philips",  "category": "Home",       "price": 199.99, "in_stock": True,  "desc": "Color-changing smart LED bulbs controllable via app or voice with millions of color options."},
    {"id": 19, "name": "Ring Video Doorbell 4",           "brand": "Ring",     "category": "Home",       "price": 199.99, "in_stock": True,  "desc": "Battery-powered smart video doorbell with HD video, color preview and two-way talk."},
    {"id": 20, "name": "Apple Watch Series 9 GPS",        "brand": "Apple",    "category": "Electronics","price": 399.00, "in_stock": True,  "desc": "Smartwatch with health sensors, fitness tracking, always-on display and crash detection."},
]
import pandas as pd
pd.DataFrame(products)[["name","brand","category","price","in_stock"]].head()

In [ ]:
# 🧠 Encode product (name + desc) for richer semantics
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
texts = [f"{p['name']} — {p['desc']}" for p in products]
vectors = encoder.encode(texts).tolist()
print("Encoded", len(vectors), "products into", len(vectors[0]), "dims")

In [ ]:
# 🚀 Create an in-memory Qdrant — no server, no Docker
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

client = QdrantClient(":memory:")
client.recreate_collection(
    collection_name="catalog",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=p["id"],
        vector=v,
        payload={k: p[k] for k in ("name","brand","category","price","in_stock")},
    )
    for p, v in zip(products, vectors)
]
client.upsert(collection_name="catalog", points=points)
print("Inserted", client.count("catalog").count, "points")

In [ ]:
# 🔍 Plain semantic search
def show(hits, query):
    print(f"\n🔎 \"{query}\"")
    rows = [{**h.payload, "score": round(h.score, 3)} for h in hits]
    return pd.DataFrame(rows)

q = "wireless headphones with great noise cancellation"
qv = encoder.encode([q]).tolist()[0]
hits = client.search(collection_name="catalog", query_vector=qv, limit=5)
show(hits, q)

In [ ]:
# 🎯 Filtered search — only IN STOCK and price < $200
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range

flt = Filter(
    must=[
        FieldCondition(key="in_stock", match=MatchValue(value=True)),
        FieldCondition(key="price",    range=Range(lt=200)),
    ]
)
q = "comfortable running shoes for daily training"
qv = encoder.encode([q]).tolist()[0]
hits = client.search(collection_name="catalog", query_vector=qv, query_filter=flt, limit=5)
show(hits, q + " [in_stock & < $200]")

In [ ]:
# 🧮 Multi-condition: brand IN list AND category match
flt = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value="Electronics")),
    ],
    should=[
        FieldCondition(key="brand", match=MatchValue(value="Apple")),
        FieldCondition(key="brand", match=MatchValue(value="Sony")),
    ],
)
q = "premium audio device for travel"
qv = encoder.encode([q]).tolist()[0]
hits = client.search(collection_name="catalog", query_vector=qv, query_filter=flt, limit=5)
show(hits, q + " [Electronics, Apple|Sony]")

In [ ]:
# 🚫 Negative filter — exclude a brand
flt = Filter(must_not=[FieldCondition(key="brand", match=MatchValue(value="Nike"))])
q = "athletic running shoes"
qv = encoder.encode([q]).tolist()[0]
hits = client.search(collection_name="catalog", query_vector=qv, query_filter=flt, limit=5)
show(hits, q + " [excluding Nike]")

## 🏋️ Exercises
1. Add a payload field `rating` (1–5 stars) to each product. Filter for `rating >= 4`.
2. Use `client.scroll(...)` to paginate through all `Electronics` products without a vector query.
3. Replace `:memory:` with `QdrantClient(path="./qdrant_local")` for disk persistence and reload.

## 🎓 Recap — When to use Qdrant
✅ Need rich filters (range, must/should/must_not, geo)
✅ Self-hosted + open source + Rust performance
❌ Want fully-managed zero-ops → use Pinecone